# Using the Our World In Data API

For every chart at Our World in Data you can download the data as CSV and accompanying metadata as JSON. This notebook demonstrates how to do this.

Here are the relevant URLs:
- https://ourworldindata.org/grapher/life-expectancy is the URL of the chart that you would visit in your browser
- https://ourworldindata.org/grapher/life-expectancy.csv is the URL of the data as CSV
- https://ourworldindata.org/grapher/life-expectancy.metadata.json is the URL of the metadata that contains information like the chart title, units, the default selection and so on

In [1]:
import pandas as pd
import requests

# Fetch the data. The User-Agent header is necessary.
df = pd.read_csv("https://ourworldindata.org/grapher/life-expectancy.csv?v=1&csvType=full&useColumnShortNames=true", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})

# Fetch the metadata
metadata = requests.get("https://ourworldindata.org/grapher/life-expectancy.metadata.json?v=1&csvType=full&useColumnShortNames=true").json()

In [2]:
df.head()

,Entity,Code,Year,life_expectancy_0__sex_total__age_0
0,Afghanistan,AFG,1950,28.1563
1,Afghanistan,AFG,1951,28.5836
2,Afghanistan,AFG,1952,29.0138
3,Afghanistan,AFG,1953,29.4521
4,Afghanistan,AFG,1954,29.6975


In [ ]:
for good_to_know in metadata["columns"]["life_expectancy_0__sex_all__age_0"]["descriptionKey"]:
  print(good_to_know)

Period life expectancy is a metric that summarizes death rates across all age groups in one particular year.
For a given year, it represents the remaining average lifespan for a hypothetical group of people, if they experienced the same age-specific death rates throughout the rest of their lives as the age-specific death rates seen in that particular year.
Prior to 1950, we use HMD (2023) data combined with Zijdeman (2015). From 1950 onwards, we use UN WPP (2022) data. For old regional data, we use Riley (2005) estimates.


Here we try to answer the question "Which countries had the biggest increase in life expectancy between 1980 and 2010?"

In [ ]:
# Filter the dataframe for the years 1980 and 2010
filtered_df = df[(df['Year'] == 1980) | (df['Year'] == 2010)]

# Pivot the data to get 1980 and 2010 life expectancy side by side for each country
pivot_df = filtered_df.pivot(index='Entity', columns='Year', values='life_expectancy_0__sex_all__age_0')

# Calculate the life expectancy increase between 1980 and 2010
pivot_df['increase'] = pivot_df[2010] - pivot_df[1980]

# Get the top 10 countries with the biggest increase in life expectancy
top_10_increase = pivot_df.sort_values(by='increase', ascending=False).head(10)

top_10_increase

Year,1980,2010,increase
Entity,,,
East Timor,28.4460,65.3036,36.8576
Western Sahara,41.7586,67.4926,25.7340
El Salvador,47.9749,71.8478,23.8729
Falkland Islands,53.9136,77.7223,23.8087
Maldives,55.6212,77.6573,22.0361
Afghanistan,39.6181,60.8508,21.2327
Algeria,53.2609,73.8081,20.5472
Bhutan,48.2179,68.4298,20.2119
Cambodia,47.5696,67.7123,20.1427


In [ ]:
print(f"""This notebook uses data from the "{metadata["chart"]["title"]}" chart at Our World In Data. The source of the data is "{metadata["chart"]["citation"]}". Last updated: {metadata["columns"]["life_expectancy_0__sex_all__age_0"]["lastUpdated"]}""")

This notebook uses data from the "Life expectancy" chart at Our World In Data. The source of the data is "UN WPP (2022); HMD (2023); Zijdeman et al. (2015); Riley (2005)". Last updated: 2023-10-10


Below you can see the full contents of the metadata

In [ ]:
metadata

{'chart': {'title': 'Life expectancy',
  'subtitle': 'The [period life expectancy](#dod:period-life-expectancy) at birth, in a given year.',
  'citation': 'UN WPP (2022); HMD (2023); Zijdeman et al. (2015); Riley (2005)',
  'originalChartUrl': 'https://ourworldindata.org/grapher/life-expectancy?v=1&csvType=full&useColumnShortNames=true',
  'selection': ['World', 'Americas', 'Europe', 'Africa', 'Asia', 'Oceania']},
 'columns': {'life_expectancy_0__sex_all__age_0': {'titleShort': 'Life expectancy at birth',
   'titleLong': 'Life expectancy at birth - Various sources – period tables',
   'descriptionShort': 'The period life expectancy at birth, in a given year.',
   'descriptionKey': ['Period life expectancy is a metric that summarizes death rates across all age groups in one particular year.',
    'For a given year, it represents the remaining average lifespan for a hypothetical group of people, if they experienced the same age-specific death rates throughout the rest of their lives as t